# Mini-protein insolubility — quickstart

Created 2026-11-08. Insoluble = 1. Run from the repository or notebooks folder. Install `pip install -e ".[dev,esm]"` first. Full fitting is opt-in; the default cells only inspect public data and model topology.

In [ ]:
from pathlib import Path
import sys, json
root = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
from solubility_public_data.data import read_records
from solubility_public_data.models import FusionModel
records = read_records(root / 'data/public/psi_records_20261108.xlsx', training=True)
print({'n': len(records), 'insoluble': sum(r['insoluble'] for r in records), 'groups': len({r['group_id'] for r in records})})

## Two public heads

Frozen ESM650 mean/max inputs; BiLSTM also reads frozen residue states. The source notebook documents every module. Structure profiles require measured or predicted structure features supplied separately.

In [ ]:
for architecture in ['residual_mlp', 'bilstm']:
    model = FusionModel(architecture)
    print(architecture, sum(p.numel() for p in model.parameters()))
    print(model)

## Download the feature cache

This verifies archive and member hashes. It downloads a large public asset only when enabled.

In [ ]:
import subprocess
DOWNLOAD_FEATURES = False
if DOWNLOAD_FEATURES:
    subprocess.run([sys.executable, 'scripts/download_features.py'], cwd=root, check=True)

## Prespecified five-seed evaluation

Both random and family-grouped splits; three-fold OOF within each training partition. Full fitting may take substantial CPU time. External labels are used only for scoring.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    for architecture in ['residual_mlp', 'bilstm']:
        subprocess.run([sys.executable, '-m', 'solubility_public_data.cli', 'evaluate', '--records', 'data/public/psi_records_20261108.xlsx', '--features', 'features/public', '--config', f'configs/{architecture}_public.json', '--splits', 'data/splits/psi_five_seed_splits_20261108.json', '--external', 'data/public/garcia_records_20261108.xlsx', 'data/public/nesg_records_20261108.xlsx', 'data/public/esol_records_20261108.xlsx', '--output', f'runs/{architecture}_20261108'], cwd=root, check=True)

## Deployment

Use `solubility fit-final` after evaluating the chosen architecture. This fits one model and sets its cutoff from full-training grouped OOF scores. Use `solubility predict` for new sequences. See `training_instruction.md` for internal structure schema and threshold interpretation.